# Regularized Hartley Neural Operator (RHNO)
### HNO vs RHNO Comparison Experiments

**Paper:** *When Real Beats Complex: Hartley Neural Operators and the Role of Spectral Basis Alignment in PDE Learning*  
**Repo:** https://github.com/jaysulk/RFHT

This notebook:
1. Clones the repo and mounts Google Drive for result persistence
2. Runs the parameter count analysis (no GPU needed)
3. Trains HNO (dense, paper baseline) vs RHNO (butterfly-factorized) on Poisson and Heat equations
4. Saves all results to Drive as `.pt` (model checkpoints) and `.pkl` (histories/metrics)
5. Plots training curves and error comparison

> **Runtime:** Set to GPU (T4 recommended). Runtime → Change runtime type → T4 GPU


## 0. Environment Check

In [ ]:
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print("GPU:", result.stdout.strip())
else:
    print("⚠️  No GPU detected — training will be slow. Switch runtime to GPU.")

# Check PyTorch
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/RHNO_experiments'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Results will be saved to: {SAVE_DIR}")


## 2. Clone Repository

In [ ]:
import os

REPO_DIR = '/content/RFHT'

if os.path.exists(REPO_DIR):
    print("Repo already cloned, pulling latest...")
    os.chdir(REPO_DIR)
    os.system('git pull')
else:
    os.system('git clone https://github.com/jaysulk/RFHT.git')

os.chdir(REPO_DIR)
print("\nRepo contents:")
for f in sorted(os.listdir('.')):
    print(f"  {f}")

# Add repo root to path (flat structure — all modules at root)
sys.path.insert(0, REPO_DIR)


In [ ]:
# Fix import paths: rhno.py was written with subpackage paths
# but the repo is flat. Patch in-place after cloning.
patches = {
    'rhno.py': [
        ('from transforms.rfht import RFHTSpectralConv2d, dht2d, idht2d',
         'from rfht import RFHTSpectralConv2d, dht2d, idht2d'),
    ],
    'training.py': [],  # no subpackage imports
}

for fname, replacements in patches.items():
    fpath = os.path.join(REPO_DIR, fname)
    if not os.path.exists(fpath):
        print(f"  Skipping {fname} (not found)")
        continue
    with open(fpath, 'r') as f:
        src = f.read()
    changed = False
    for old, new in replacements:
        if old in src:
            src = src.replace(old, new)
            changed = True
            print(f"  Patched {fname}: '{old}' -> '{new}'")
    if changed:
        with open(fpath, 'w') as f:
            f.write(src)
    else:
        print(f"  {fname}: already correct")

print("Import paths OK.")

# Also fix gradient_error in training.py (torch.norm 3-dim bug)
_ge_old = '''    return (torch.norm(gp - gt, dim=(-1, -2, -3)) /
            torch.norm(gt, dim=(-1, -2, -3))).mean()'''
_ge_new = '''    diff_norm = torch.linalg.norm(gp - gt, dim=(-3, -2, -1))
    true_norm = torch.linalg.norm(gt,      dim=(-3, -2, -1))
    return (diff_norm / (true_norm + 1e-8)).mean()'''

tr_path = os.path.join(REPO_DIR, 'training.py')
with open(tr_path, 'r') as f:
    src = f.read()
if _ge_old in src:
    src = src.replace(_ge_old, _ge_new)
    with open(tr_path, 'w') as f:
        f.write(src)
    print("  Patched training.py: gradient_error fixed")
else:
    print("  training.py: gradient_error already OK")


## 3. Imports

In [ ]:
import torch
import numpy as np
import pickle
import math
import time
from torch.utils.data import DataLoader, random_split
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, clear_output

# Repo modules (flat structure)
from rfht import (
    dht2d, idht2d, dibit_reverse_indices,
    ButterflyStage, RFHTSpectralConv2d
)
from rhno import (
    HNOSpectralConv2d, HartleyNeuralOperator,
    make_hno, make_rhno
)
from training import (
    relative_l2, gradient_error,
    make_grf, make_eigenfunction_ic, make_gaussian_bump_ic,
    solve_heat_equation, solve_poisson,
    HeatDataset, PoissonDataset,
    train_model, eval_epoch
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")


In [ ]:
# Fix gradient_error: torch.norm with 3 dims hits linalg.matrix_norm (needs 2).
# Redefine using torch.linalg.norm which handles arbitrary dims.
import torch

def gradient_error(pred, true):
    def grad(u):
        gx = (torch.roll(u, -1, -1) - torch.roll(u, 1, -1)) / 2
        gy = (torch.roll(u, -1, -2) - torch.roll(u, 1, -2)) / 2
        return torch.stack([gx, gy], dim=-1)
    gp, gt = grad(pred), grad(true)
    # flatten spatial+vector dims for norm
    diff_norm = torch.linalg.norm(gp - gt, dim=(-3, -2, -1))
    true_norm = torch.linalg.norm(gt,      dim=(-3, -2, -1))
    return (diff_norm / (true_norm + 1e-8)).mean()

# Also patch it into the training module so train_model uses the fixed version
import training as _tr
_tr.gradient_error = gradient_error

print("gradient_error patched OK")


## 4. Parameter Count Analysis

HNO spectral params scale **O(M²)** (dense weights per quadrant).  
RHNO spectral params scale **O(M log M)** (butterfly factorization).

This is the direct analogue of FFT vs naive DFT complexity.


In [ ]:
print("=" * 70)
print("Parameter Count: HNO (dense) vs RHNO (butterfly-factorized)")
print("=" * 70)

configs = [
    (8,  32, 3, "small"),
    (12, 32, 3, "paper-default"),
    (16, 32, 3, "medium"),
    (20, 48, 3, "large"),
    (12, 32, 4, "elliptic-4blk"),
]

rows = []
print(f"\n{'Config':<22} {'HNO Spec':>10} {'RHNO Spec':>10} {'Ratio':>8} {'Stages':>7}")
print("-" * 62)

for modes, width, blocks, label in configs:
    hno = make_hno(width=width, modes=modes, num_blocks=blocks)
    rhno = make_rhno(width=width, modes=modes, num_blocks=blocks)
    hc = hno.parameter_count()
    rc = rhno.parameter_count()
    stages = max(1, int(math.log2(modes)))
    ratio = rc['spectral'] / hc['spectral']
    cfg = f"{label}(m={modes},w={width})"
    print(f"{cfg:<22} {hc['spectral']:>10,} {rc['spectral']:>10,} {ratio:>8.3f} {stages:>7d}")
    rows.append((cfg, hc, rc, ratio, stages))

print()
print("Theoretical scaling (per-mode, width-normalized):")
print(f"  {'Modes':>6} {'HNO O(M²)':>12} {'RHNO O(MlogM)':>15} {'Ratio':>8}")
print("  " + "-" * 45)
for modes in [4, 8, 12, 16, 20, 32, 64]:
    hno_s = 8 * modes * modes
    stages = max(1, int(math.log2(modes)))
    rhno_s = 4 * stages * (modes // 2) * 4 + 4 * 2
    print(f"  {modes:>6} {hno_s:>12,} {rhno_s:>15,} {rhno_s/hno_s:>8.3f}")


## 5. Experiment Configuration

Adjust these to trade speed vs fidelity.  
**Quick run:** `RESOLUTION=64, N_SAMPLES=100, N_EPOCHS=50`  
**Paper setting:** `RESOLUTION=128, N_SAMPLES=200, N_EPOCHS=200`


In [ ]:
# ── Experiment config ──────────────────────────────────────────────────
RESOLUTION   = 64      # 128 for paper; 64 for quick
N_SAMPLES    = 100     # 200 for paper
N_EPOCHS     = 50      # 200 for paper
BATCH_SIZE   = 8
MODES        = 12      # spectral modes retained per quadrant edge
WIDTH        = 32      # hidden channel width
BUTTERFLY_STAGES = None  # None = log2(modes); set int to ablate depth

# PDEs and IC types to run
EXPERIMENTS = [
    # (pde,       ic_type,        num_blocks, in_channels)
    ('poisson',   'grf',          4,          3),
    ('poisson',   'gaussian_bump',4,          3),
    ('heat',      'grf',          3,          4),
]

# HNO-optimized hyperparams (Table 3 of paper)
HNO_LR          = 3.8e-3
HNO_WD          = 1e-6
HNO_CLIP        = 5.0
HNO_SCHEDULER   = 'step'

print("Configuration:")
print(f"  Resolution:   {RESOLUTION}×{RESOLUTION}")
print(f"  Samples:      {N_SAMPLES}")
print(f"  Epochs:       {N_EPOCHS}")
print(f"  Modes:        {MODES}")
print(f"  Width:        {WIDTH}")
print(f"  Butterfly:    {BUTTERFLY_STAGES or f'log2({MODES})={max(1,int(math.log2(MODES)))} stages'}")
print(f"  Experiments:  {len(EXPERIMENTS)}")
print(f"  Device:       {DEVICE}")


## 6. Run HNO vs RHNO Experiments

Each experiment:
1. Generates PDE dataset (IC → solution pairs)
2. Trains HNO (dense, paper baseline) with paper-optimized hyperparams
3. Trains RHNO (butterfly-factorized) with same hyperparams
4. Saves model checkpoints and training history to Drive


In [ ]:
all_results = {}

for pde, ic_type, num_blocks, in_channels in EXPERIMENTS:
    exp_key = f"{pde}_{ic_type}"
    print(f"\n{'='*65}")
    print(f"EXPERIMENT: {pde.upper()} / {ic_type}")
    print(f"{'='*65}")

    # ── Build dataset ──────────────────────────────────────────────────
    print("\nGenerating dataset...")
    t0 = time.time()

    if pde == 'poisson':
        dataset = PoissonDataset(
            n_samples=N_SAMPLES, resolution=RESOLUTION,
            ic_type=ic_type, device=DEVICE
        )
    elif pde == 'heat':
        dataset = HeatDataset(
            n_samples=N_SAMPLES, resolution=RESOLUTION,
            ic_type=ic_type, device=DEVICE
        )

    n_train = int(0.8 * len(dataset))
    n_test  = len(dataset) - n_train
    train_ds, test_ds = random_split(
        dataset, [n_train, n_test],
        generator=torch.Generator().manual_seed(42)
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                               shuffle=True, drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                               shuffle=False)

    print(f"  Train: {n_train}, Test: {n_test}  ({time.time()-t0:.1f}s)")

    exp_results = {}

    for model_name, use_rfht in [('HNO', False), ('RHNO', True)]:
        print(f"\n── Training {model_name} ──────────────────────────────────")
        t0 = time.time()

        model = HartleyNeuralOperator(
            in_channels=in_channels,
            out_channels=1,
            width=WIDTH,
            modes=MODES,
            num_blocks=num_blocks,
            use_rfht=use_rfht,
            num_butterfly_stages=BUTTERFLY_STAGES
        )
        counts = model.parameter_count()
        print(f"  Params: {counts['total']:,}  (spectral: {counts['spectral']:,})")

        history = train_model(
            model, train_loader, test_loader,
            n_epochs=N_EPOCHS,
            lr=HNO_LR, weight_decay=HNO_WD,
            clip_grad=HNO_CLIP,
            scheduler_type=HNO_SCHEDULER,
            device=DEVICE,
            verbose=True
        )
        elapsed = time.time() - t0
        print(f"  Training time: {elapsed:.0f}s")

        # Save checkpoint to Drive
        ckpt_path = os.path.join(SAVE_DIR, f"{exp_key}_{model_name}.pt")
        torch.save({
            'model_state_dict': model.cpu().state_dict(),
            'config': {
                'in_channels': in_channels,
                'out_channels': 1,
                'width': WIDTH,
                'modes': MODES,
                'num_blocks': num_blocks,
                'use_rfht': use_rfht,
                'num_butterfly_stages': BUTTERFLY_STAGES,
            },
            'history': history,
            'param_counts': counts,
            'pde': pde,
            'ic_type': ic_type,
            'resolution': RESOLUTION,
            'n_samples': N_SAMPLES,
            'n_epochs': N_EPOCHS,
            'training_time_s': elapsed,
        }, ckpt_path)
        print(f"  ✓ Checkpoint saved: {ckpt_path}")

        exp_results[model_name] = {
            'history': history,
            'counts': counts,
            'elapsed': elapsed,
        }

    all_results[exp_key] = exp_results

    # Quick summary
    hno_best  = exp_results['HNO']['history']['best_test_rel_l2']
    rhno_best = exp_results['RHNO']['history']['best_test_rel_l2']
    ratio     = rhno_best / hno_best
    print(f"\n  SUMMARY [{exp_key}]")
    print(f"    HNO  best Rel-L2: {hno_best:.6f}  "
          f"({exp_results['HNO']['counts']['total']:,} params)")
    print(f"    RHNO best Rel-L2: {rhno_best:.6f}  "
          f"({exp_results['RHNO']['counts']['total']:,} params)")
    print(f"    RHNO/HNO error ratio:  {ratio:.3f}x")
    print(f"    RHNO/HNO param ratio:  "
          f"{exp_results['RHNO']['counts']['total']/exp_results['HNO']['counts']['total']:.3f}x")

# Save all results summary as pkl
summary_path = os.path.join(SAVE_DIR, 'all_results_summary.pkl')
with open(summary_path, 'wb') as f:
    pickle.dump(all_results, f)
print(f"\n✓ Full results summary saved: {summary_path}")


## 7. Results Table

In [ ]:
print("\n" + "="*75)
print("RESULTS SUMMARY — HNO vs RHNO")
print("="*75)
print(f"{'Experiment':<22} {'Model':<6} {'Best Rel-L2':>12} {'Params':>10} "
      f"{'Spec Params':>12} {'RHNO/HNO':>10}")
print("-"*75)

for exp_key, exp_res in all_results.items():
    pde, ic = exp_key.split('_', 1)
    hno_best  = exp_res['HNO']['history']['best_test_rel_l2']
    rhno_best = exp_res['RHNO']['history']['best_test_rel_l2']

    for model_name in ['HNO', 'RHNO']:
        res = exp_res[model_name]
        best = res['history']['best_test_rel_l2']
        ratio_str = f"{rhno_best/hno_best:.3f}x" if model_name == 'RHNO' else "—"
        label = f"{pde}/{ic}" if model_name == 'HNO' else ""
        print(f"{label:<22} {model_name:<6} {best:>12.6f} "
              f"{res['counts']['total']:>10,} "
              f"{res['counts']['spectral']:>12,} "
              f"{ratio_str:>10}")

print()
print("RHNO/HNO < 1.0 = RHNO outperforms with fewer parameters")


## 8. Training Curves

In [ ]:
n_exp = len(all_results)
fig, axes = plt.subplots(n_exp, 2, figsize=(14, 4 * n_exp))
if n_exp == 1:
    axes = [axes]

colors = {'HNO': '#E05C5C', 'RHNO': '#4A90D9'}

for row, (exp_key, exp_res) in enumerate(all_results.items()):
    pde, ic = exp_key.split('_', 1)
    ax_loss, ax_test = axes[row]

    for model_name, res in exp_res.items():
        h = res['history']
        epochs = range(10, N_EPOCHS + 1, 10)
        epochs = list(epochs)[:len(h['train_loss'])]

        ax_loss.semilogy(epochs, h['train_loss'],
                          color=colors[model_name], alpha=0.4,
                          linestyle='--', linewidth=1)
        ax_test.semilogy(epochs, h['test_rel_l2'],
                          color=colors[model_name],
                          label=f"{model_name} ({res['counts']['total']:,} params)",
                          linewidth=2)

    ax_loss.set_title(f"{pde.upper()} / {ic} — Train MSE (log)")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Train Loss")
    ax_loss.grid(True, alpha=0.3)

    ax_test.set_title(f"{pde.upper()} / {ic} — Test Rel-L² (log)")
    ax_test.set_xlabel("Epoch")
    ax_test.set_ylabel("Rel-L² Error")
    ax_test.legend(fontsize=9)
    ax_test.grid(True, alpha=0.3)

plt.suptitle("HNO (dense) vs RHNO (butterfly-factorized)", fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()

plot_path = os.path.join(SAVE_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Plot saved: {plot_path}")


## 9. Parameter Efficiency: Error vs Params

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

markers = {'HNO': 'o', 'RHNO': 's'}
sizes   = {'HNO': 100, 'RHNO': 100}

for exp_key, exp_res in all_results.items():
    pde, ic = exp_key.split('_', 1)
    label_base = f"{pde}/{ic}"

    for model_name, res in exp_res.items():
        best = res['history']['best_test_rel_l2']
        params = res['counts']['total']
        ax.scatter(params, best,
                   marker=markers[model_name],
                   s=sizes[model_name],
                   color=colors[model_name],
                   label=f"{model_name} – {label_base}",
                   zorder=3)
        ax.annotate(f"{model_name}\n{label_base}",
                    (params, best),
                    textcoords="offset points", xytext=(5, 5),
                    fontsize=7, alpha=0.8)

# Connect HNO→RHNO pairs
for exp_key, exp_res in all_results.items():
    hno_p  = exp_res['HNO']['counts']['total']
    hno_e  = exp_res['HNO']['history']['best_test_rel_l2']
    rhno_p = exp_res['RHNO']['counts']['total']
    rhno_e = exp_res['RHNO']['history']['best_test_rel_l2']
    ax.annotate('', xy=(rhno_p, rhno_e), xytext=(hno_p, hno_e),
                arrowprops=dict(arrowstyle='->', color='gray',
                                lw=1.2, alpha=0.5))

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel("Total Parameters", fontsize=11)
ax.set_ylabel("Best Test Rel-L² Error", fontsize=11)
ax.set_title("Parameter Efficiency: HNO vs RHNO\n"
             "(lower-left = better; arrows show HNO→RHNO reduction)",
             fontsize=11)
ax.grid(True, which='both', alpha=0.2)

# De-duplicate legend
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), fontsize=8,
          loc='upper right')

plt.tight_layout()
eff_path = os.path.join(SAVE_DIR, 'param_efficiency.png')
plt.savefig(eff_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Plot saved: {eff_path}")


## 10. Butterfly Depth Ablation  *(optional — runs additional training)*

Ablates the number of butterfly stages on the first experiment.  
Jones (2022) uses log₄(M) = log₂(M)/2 stages for radix-4 efficiency.  
This tests whether shallower = more regularized = better or worse.


In [ ]:
RUN_ABLATION = False  # Set True to run (takes ~2-3× training time)

if RUN_ABLATION:
    pde, ic_type, num_blocks, in_channels = EXPERIMENTS[0]
    exp_key = f"{pde}_{ic_type}"

    dataset = PoissonDataset(n_samples=N_SAMPLES, resolution=RESOLUTION,
                              ic_type=ic_type, device=DEVICE)
    n_train = int(0.8 * len(dataset))
    train_ds, test_ds = random_split(
        dataset, [n_train, len(dataset)-n_train],
        generator=torch.Generator().manual_seed(42)
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                               shuffle=True, drop_last=True)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    ablation_results = {}
    max_stages = max(1, int(math.log2(MODES)))

    for stages in range(1, max_stages + 1):
        print(f"\n── RHNO depth ablation: {stages} stage(s) ──────────────")
        model = make_rhno(in_channels=in_channels, width=WIDTH,
                           modes=MODES, num_blocks=num_blocks,
                           num_butterfly_stages=stages)
        counts = model.parameter_count()
        print(f"  Params: {counts['total']:,}")
        history = train_model(model, train_loader, test_loader,
                               n_epochs=N_EPOCHS, lr=HNO_LR,
                               weight_decay=HNO_WD, clip_grad=HNO_CLIP,
                               scheduler_type=HNO_SCHEDULER,
                               device=DEVICE, verbose=False)
        ablation_results[stages] = {
            'history': history, 'counts': counts
        }
        print(f"  Best Rel-L2: {history['best_test_rel_l2']:.6f}")

    ablation_path = os.path.join(SAVE_DIR, f'ablation_depth_{exp_key}.pkl')
    with open(ablation_path, 'wb') as f:
        pickle.dump(ablation_results, f)
    print(f"\n✓ Ablation results saved: {ablation_path}")

    # Plot
    stages_list = sorted(ablation_results.keys())
    errors  = [ablation_results[s]['history']['best_test_rel_l2'] for s in stages_list]
    params  = [ablation_results[s]['counts']['total'] for s in stages_list]
    hno_ref = all_results[exp_key]['HNO']['history']['best_test_rel_l2']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(stages_list, errors, 'o-', color='#4A90D9', linewidth=2, markersize=8)
    ax1.axhline(hno_ref, color='#E05C5C', linestyle='--', linewidth=1.5,
                label=f'HNO baseline ({hno_ref:.4f})')
    ax1.set_xlabel("Butterfly Stages")
    ax1.set_ylabel("Best Test Rel-L²")
    ax1.set_title(f"Depth Ablation: {pde.upper()}/{ic_type}")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(stages_list, params, 's-', color='#4A90D9', linewidth=2, markersize=8)
    ax2.axhline(all_results[exp_key]['HNO']['counts']['total'],
                color='#E05C5C', linestyle='--', linewidth=1.5, label='HNO params')
    ax2.set_xlabel("Butterfly Stages")
    ax2.set_ylabel("Total Parameters")
    ax2.set_title("Parameter Count vs Depth")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'ablation_depth.png'), dpi=150)
    plt.show()
else:
    print("Skipped (set RUN_ABLATION = True to run).")


## 11. Saved Files Summary

In [ ]:
import os

print(f"Files saved to {SAVE_DIR}:\n")
total_bytes = 0
for fname in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, fname)
    size  = os.path.getsize(fpath)
    total_bytes += size
    ext   = fname.split('.')[-1].upper()
    print(f"  [{ext:4s}] {fname:<45}  {size/1024/1024:.2f} MB")

print(f"\nTotal: {total_bytes/1024/1024:.2f} MB")
print()
print("To reload a checkpoint later:")
print("  ckpt = torch.load('/content/drive/MyDrive/RHNO_experiments/poisson_grf_HNO.pt')")
print("  model = HartleyNeuralOperator(**ckpt['config'])")
print("  model.load_state_dict(ckpt['model_state_dict'])")
print()
print("To reload results summary:")
print("  with open('.../all_results_summary.pkl', 'rb') as f:")
print("      results = pickle.load(f)")
